# Single vs Batch Predict Benchmark (Sweet Spot)

This notebook compares:
- `POST /anonymizer/predict` (single paragraph per request)
- `POST /anonymizer/predict-batch` (multiple paragraphs per request)

It benchmarks latency and throughput and suggests a **sweet spot** batch size.


In [ ]:
from __future__ import annotations

from collections import defaultdict
from statistics import mean, median
from time import perf_counter
import math
import os
from pathlib import Path

import requests
from tqdm import tqdm

from aymurai.experiments.entity_disambiguation.runner import (
    call_extraction_api as extract_document,
)


In [ ]:
# Endpoint and benchmark config
API_URL = "http://localhost:8000"
USE_CACHE = False
TIMEOUT_S = 180

# Real document extraction settings
DATA_ROOT = Path(
    os.getenv(
        "DISAMBIGUATION_DATA_ROOT",
        "../../../resources/data/restricted/disambiguation-eval/files",
    )
)
DOC_EXTENSIONS = {".pdf", ".docx"}
MAX_FILES = None

# How many benchmark repetitions per setting
REPEATS = 5

# Candidate client-side batch sizes for /predict-batch
CLIENT_BATCH_SIZES = [1, 2, 4, 8, 16, 32, 64]

# Maximum number of extracted paragraphs used in benchmark (per file)
TARGET_TOTAL_PARAGRAPHS = 500


In [ ]:
def discover_documents(root: Path, extensions: set[str]) -> list[Path]:
    extensions = {ext.lower() for ext in extensions}
    return sorted(
        path
        for path in root.rglob("*")
        if path.is_file() and path.suffix.lower() in extensions
    )


documents = discover_documents(DATA_ROOT, DOC_EXTENSIONS)
if MAX_FILES:
    documents = documents[:MAX_FILES]

print(f"Found {len(documents)} documents for benchmark")
if not documents:
    raise ValueError(f"No documents found in {DATA_ROOT}")

session = requests.Session()


In [ ]:
def chunked(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i : i + size]


def extract_paragraphs_for_doc(doc_path: Path) -> list[str]:
    document = extract_document(
        session,
        endpoint=f"{API_URL}/misc/document-extract",
        file_path=doc_path,
        timeout_s=300,
    )
    paragraphs = document["detail"]["document"] or []
    return paragraphs[:TARGET_TOTAL_PARAGRAPHS]


def call_predict_single(paragraphs: list[str]) -> tuple[float, int]:
    start = perf_counter()
    processed = 0
    for p in paragraphs:
        r = session.post(
            url=f"{API_URL}/anonymizer/predict",
            json={"text": p},
            params={"use_cache": USE_CACHE},
            timeout=TIMEOUT_S,
        )
        r.raise_for_status()
        _ = r.json()
        processed += 1
    elapsed = perf_counter() - start
    return elapsed, processed


def call_predict_batch(paragraphs: list[str], client_batch_size: int) -> tuple[float, int]:
    start = perf_counter()
    processed = 0
    for chunk in chunked(paragraphs, client_batch_size):
        payload = [{"text": p} for p in chunk]
        r = session.post(
            url=f"{API_URL}/anonymizer/predict-batch",
            json=payload,
            params={"use_cache": USE_CACHE},
            timeout=TIMEOUT_S,
        )
        r.raise_for_status()
        data = r.json().get("data", [])
        processed += len(data)
    elapsed = perf_counter() - start
    return elapsed, processed


In [ ]:
rows = []
skipped_docs = []

for doc_path in tqdm(documents, desc="Documents", unit="doc"):
    try:
        paragraphs = extract_paragraphs_for_doc(doc_path)
    except Exception as exc:
        skipped_docs.append((str(doc_path), f"extract_error: {exc}"))
        continue

    if not paragraphs:
        skipped_docs.append((str(doc_path), "0_paragraphs"))
        continue

    for run in range(1, REPEATS + 1):
        elapsed, processed = call_predict_single(paragraphs)
        rows.append(
            {
                "document": str(doc_path),
                "mode": "single",
                "client_batch_size": 1,
                "run": run,
                "processed": processed,
                "total_s": elapsed,
                "ms_per_paragraph": (elapsed / processed) * 1000 if processed else None,
                "paragraphs_per_s": processed / elapsed if elapsed else None,
            }
        )

    for batch_size in CLIENT_BATCH_SIZES:
        for run in range(1, REPEATS + 1):
            elapsed, processed = call_predict_batch(paragraphs, client_batch_size=batch_size)
            rows.append(
                {
                    "document": str(doc_path),
                    "mode": "batch",
                    "client_batch_size": batch_size,
                    "run": run,
                    "processed": processed,
                    "total_s": elapsed,
                    "ms_per_paragraph": (elapsed / processed) * 1000 if processed else None,
                    "paragraphs_per_s": processed / elapsed if elapsed else None,
                }
            )

print(f"Collected rows: {len(rows)}")
print(f"Skipped docs: {len(skipped_docs)}")
if skipped_docs:
    for path, reason in skipped_docs[:10]:
        print(f"- {path}: {reason}")

if not rows:
    raise RuntimeError("No benchmark rows collected")


In [ ]:
by_key = defaultdict(list)
for row in rows:
    key = (row["document"], row["mode"], row["client_batch_size"])
    by_key[key].append(row)

summary = []
for (document, mode, bs), group in sorted(by_key.items(), key=lambda x: (x[0][0], x[0][1], x[0][2])):
    total_s_values = [g["total_s"] for g in group]
    mpp_values = [g["ms_per_paragraph"] for g in group if g["ms_per_paragraph"] is not None]
    tps_values = [g["paragraphs_per_s"] for g in group if g["paragraphs_per_s"] is not None]

    summary.append(
        {
            "document": document,
            "mode": mode,
            "client_batch_size": bs,
            "runs": len(group),
            "mean_total_s": mean(total_s_values),
            "median_total_s": median(total_s_values),
            "mean_ms_per_paragraph": mean(mpp_values),
            "mean_paragraphs_per_s": mean(tps_values),
        }
    )

header = (
    f"{'document':<32} {'mode':<8} {'batch':>6} {'runs':>5} {'mean_total_s':>14} "
    f"{'mean_ms/paragraph':>20} {'mean_paragraphs/s':>19}"
)
print(header)
print('-' * len(header))
for s in summary:
    name = Path(s['document']).name[:32]
    print(
        f"{name:<32} {s['mode']:<8} {s['client_batch_size']:>6} {s['runs']:>5} "
        f"{s['mean_total_s']:>14.3f} {s['mean_ms_per_paragraph']:>20.2f} {s['mean_paragraphs_per_s']:>19.2f}"
    )


In [ ]:
# Per-file sweet spot + global production recommendation
per_file = defaultdict(list)
for s in summary:
    per_file[s['document']].append(s)

per_file_reco = []
for document, entries in sorted(per_file.items()):
    single = next((e for e in entries if e['mode'] == 'single'), None)
    batch_entries = [e for e in entries if e['mode'] == 'batch']
    if not single or not batch_entries:
        continue

    best_tps = max(e['mean_paragraphs_per_s'] for e in batch_entries)
    threshold = best_tps * 0.95
    candidates = [e for e in batch_entries if e['mean_paragraphs_per_s'] >= threshold]
    sweet_spot = min(candidates, key=lambda e: e['client_batch_size'])
    speedup = sweet_spot['mean_paragraphs_per_s'] / single['mean_paragraphs_per_s']

    per_file_reco.append(
        {
            'document': document,
            'sweet_spot_batch': sweet_spot['client_batch_size'],
            'sweet_spot_tps': sweet_spot['mean_paragraphs_per_s'],
            'single_tps': single['mean_paragraphs_per_s'],
            'speedup_vs_single': speedup,
        }
    )

if not per_file_reco:
    raise RuntimeError('No per-file recommendations could be computed')

print('Per-file sweet spot (95% of best throughput):')
for r in per_file_reco:
    print(
        f"- {Path(r['document']).name}: batch={r['sweet_spot_batch']} | "
        f"sweet_spot_tps={r['sweet_spot_tps']:.2f} | speedup_vs_single={r['speedup_vs_single']:.2f}x"
    )

# Aggregate across files for production default
batch_perf = defaultdict(list)
for s in summary:
    if s['mode'] == 'batch':
        batch_perf[s['client_batch_size']].append(s['mean_paragraphs_per_s'])

batch_agg = []
for bs in sorted(batch_perf):
    values = batch_perf[bs]
    batch_agg.append(
        {
            'client_batch_size': bs,
            'files': len(values),
            'mean_tps': mean(values),
            'median_tps': median(values),
        }
    )

best_mean_tps = max(x['mean_tps'] for x in batch_agg)
mean_threshold = best_mean_tps * 0.95
global_candidates = [x for x in batch_agg if x['mean_tps'] >= mean_threshold]
recommended = min(global_candidates, key=lambda x: x['client_batch_size'])

files_supporting_reco = sum(1 for r in per_file_reco if r['sweet_spot_batch'] <= recommended['client_batch_size'])
support_ratio = files_supporting_reco / len(per_file_reco)

print('\nAggregate batch throughput across files:')
for x in batch_agg:
    print(
        f"- batch={x['client_batch_size']}: files={x['files']}, "
        f"mean_tps={x['mean_tps']:.2f}, median_tps={x['median_tps']:.2f}"
    )

print('\nProduction recommendation:')
print(
    f"Use client batch size {recommended['client_batch_size']} as default. "
    f"It is within 95% of the best average throughput ({recommended['mean_tps']:.2f} tps vs best {best_mean_tps:.2f} tps)."
)
print(f"Coverage signal: {files_supporting_reco}/{len(per_file_reco)} files have sweet spot <= recommended batch size ({support_ratio:.0%}).")


In [ ]:
# Export benchmark artifacts to CSV
import csv

OUTPUT_DIR = Path('outputs/anonymization-benchmark')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def write_csv(path: Path, data: list[dict]):
    if not data:
        print(f'Skipping {path.name}: no rows')
        return
    fieldnames = list(data[0].keys())
    with path.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(data)
    print(f'Wrote {len(data)} rows -> {path}')

write_csv(OUTPUT_DIR / 'rows.csv', rows)
write_csv(OUTPUT_DIR / 'summary.csv', summary)
write_csv(OUTPUT_DIR / 'per_file_reco.csv', per_file_reco)
write_csv(OUTPUT_DIR / 'batch_agg.csv', batch_agg)

if skipped_docs:
    skipped_rows = [{'document': d, 'reason': reason} for d, reason in skipped_docs]
    write_csv(OUTPUT_DIR / 'skipped_docs.csv', skipped_rows)
